# Chapter 3 – Modal analysis (lecture companion)
**Vibrations – Centrale Nantes**

Interactive figures for chapter 3 of the lecture:
* **Figure 1** – Rayleigh damping: modal damping ratio versus frequency, and its effect on the free response.

**How to use it:** run all the cells (*Run → Run all cells*). In the interactive figures, move the **sliders** to change the parameters and use the **Play/Pause** buttons below the figure.
The code cells are hidden (click on them to show the code).

In [ ]:
#@title Setup (run first): interactive figures with sliders
import sys, subprocess
if "pyodide" in sys.modules:                     # JupyterLite (in the browser)
    import piplite
    await piplite.install(["ipympl", "ipywidgets"])
elif "google.colab" in sys.modules:              # Google Colab
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ipympl"])
    from google.colab import output
    output.enable_custom_widget_manager()
try:
    import ipympl
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    print("ipympl is not installed: the animations will be shown without sliders (pip install ipympl).")

In [ ]:
#@title Helper functions (run this cell first)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

plt.rcParams.update({"figure.dpi": 90, "animation.embed_limit": 60, "font.size": 10,
                     # LaTeX-like rendering without a LaTeX installation: $...$ formulas and text in STIX (Times-like)
                     "mathtext.fontset": "stix", "font.family": "STIXGeneral"})
BLUE, RED, GREEN, GREY = "#1f4e99", "#a3201d", "#2e6b30", "#777777"


def rk4(f, y0, t):
    """Fixed-step Runge-Kutta 4 integration of dy/dt = f(t, y)."""
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for i in range(len(t) - 1):
        h = t[i + 1] - t[i]
        k1 = f(t[i], y[i])
        k2 = f(t[i] + h / 2, y[i] + h / 2 * k1)
        k3 = f(t[i] + h / 2, y[i] + h / 2 * k2)
        k4 = f(t[i] + h, y[i] + h * k3)
        y[i + 1] = y[i] + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return y


def spring_xy(p0, p1, n=7, width=0.012, lead=0.15):
    """Zig-zag polyline of a spring between points p0 and p1."""
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    L = np.linalg.norm(p1 - p0)
    u = (p1 - p0) / L
    v = np.array([-u[1], u[0]])
    pts = [p0, p0 + lead * L * u]
    for i in range(2 * n):
        s = lead * L + (1 - 2 * lead) * L * (i + 0.5) / (2 * n)
        pts.append(p0 + s * u + (width if i % 2 == 0 else -width) * v)
    pts += [p1 - lead * L * u, p1]
    pts = np.array(pts)
    return pts[:, 0], pts[:, 1]


def hatch(ax, x0, x1, y, down=True, n=12, size=0.012):
    """Draw a fixed support (line + hatching)."""
    ax.plot([x0, x1], [y, y], color="k", lw=2)
    s = -size if down else size
    for x in np.linspace(x0, x1, n):
        ax.plot([x, x + size], [y, y - s], color="k", lw=0.8)


def make_anim(fig, update, nframes, fps=25):
    """Build an animation and remember what is needed to export it."""
    anim = animation.FuncAnimation(fig, update, frames=nframes, interval=1000 / fps, blit=False)
    anim.update_func, anim.nframes = update, nframes
    return anim


def show(anim):
    """Display an animation in the notebook (works in Jupyter, JupyterLite and Colab)."""
    plt.close(anim._fig)
    display(HTML(anim.to_jshtml()))


def export_frames(anim, name, folder="gifs", dpi=120):
    """Teacher only: save the frames as name-0.png, name-1.png, ... for the LaTeX animate package."""
    import os
    os.makedirs(f"{folder}/{name}", exist_ok=True)
    for i in range(anim.nframes):
        anim.update_func(i)
        anim._fig.savefig(f"{folder}/{name}/{name}-{i}.png", dpi=dpi)
    print(f"{anim.nframes} frames saved in {folder}/{name}/ (last index {anim.nframes - 1})")


# ---------------------------------------------------------------------------
# Live animation: sliders drawn on the figure + Play/Pause widget below it.
# Needs ipympl (Jupyter, JupyterLite, VS Code, Colab). The time steps are driven
# by the ipywidgets "Play" widget (timer in the browser), which also works in
# JupyterLite where matplotlib timers do not run.
# Without ipympl, the animation is shown with the default parameters.
# ---------------------------------------------------------------------------
from matplotlib.widgets import Slider

INTERACTIVE = "ipympl" in plt.get_backend().lower() or "widget" in plt.get_backend().lower()


class LiveAnimation:
    """
    sliders : list of (name, label, vmin, vmax, vinit, step)
    compute : function(params dict) -> data dict (solution on a fine time grid)
    setup   : function(data) -> None, updates the static elements (curves, limits...)
    frame   : function(data, t) -> None, updates the moving elements at time t
    T       : simulated duration (s), played in a loop
              (compute may return data["T"] to change it, e.g. a whole number of periods)
    slow    : slow motion factor
    """

    def __init__(self, fig, sliders, compute, setup, frame, T, slow=1, fps=20):
        self.fig, self.compute, self.setup, self.frame = fig, compute, setup, frame
        self.T, self.slow, self.fps = T, slow, fps
        n = len(sliders)
        fig.subplots_adjust(bottom=0.08 + 0.045 * n)
        self.sliders = {}
        for j, (name, label, vmin, vmax, vinit, step) in enumerate(sliders):
            ax = fig.add_axes([0.22, 0.02 + 0.045 * (n - 1 - j), 0.5, 0.025])
            s = Slider(ax, label, vmin, vmax, valinit=vinit, valstep=step)
            s.on_changed(self._changed)
            self.sliders[name] = s
        for a in fig.axes:                        # drawings with equal aspect: keep them aligned at the top
            if a.get_aspect() == 1.0:
                a.set_anchor("N")
        self.i = 0
        self.data = self.compute(self.params())
        self._loop()
        self.setup(self.data)
        self.frame(self.data, 0.0)
        if INTERACTIVE:
            import ipywidgets as W
            fig.canvas.header_visible = False
            # Play button (timer in the browser) linked to a time slider: the slider can also be moved by hand
            self.play = W.Play(value=0, min=0, max=self.nframes - 1, step=1,
                               interval=int(1000 / fps), repeat=True, description="Play")
            self.tslider = W.IntSlider(value=0, min=0, max=self.nframes - 1, readout=False,
                                       layout=W.Layout(width="45%"))
            W.jslink((self.play, "value"), (self.tslider, "value"))
            self.tlabel = W.Label()
            self.out = W.Output()                 # error messages of the callbacks (if any)
            self.tslider.observe(self._tick, names="value")
            plt.show()                            # figure first, controls below
            display(W.VBox([W.HBox([self.play, self.tslider, self.tlabel]), self.out]))   # paused by default
            self._show_t()
        else:
            print("Static version (default parameters): install ipympl to use the sliders.")
            show(make_anim(fig, lambda i: self.frame(self.data, i / self.nframes * self.Tloop), self.nframes, fps))

    def params(self):
        return {name: s.val for name, s in self.sliders.items()}

    def _loop(self):
        """Loop duration (from compute if given) and number of frames of one loop."""
        self.Tloop = self.data.get("T", self.T)
        self.nframes = max(2, int(round(self.Tloop * self.fps * self.slow)))

    def _t(self):
        return (self.i % self.nframes) / self.nframes * self.Tloop

    def _show_t(self):
        self.tlabel.value = f"t = {self._t():.2f} s"

    def _report(self, err):
        import traceback
        with self.out:
            traceback.print_exc()

    def _changed(self, _):
        try:
            self.data = self.compute(self.params())
            self._loop()
            if hasattr(self, "play"):
                self.i = min(self.i, self.nframes - 1)
                self.play.max = self.tslider.max = self.nframes - 1
                self._show_t()
            self.setup(self.data)
            self.frame(self.data, self._t())
            self.fig.canvas.draw_idle()
        except Exception as err:
            self._report(err)

    def _tick(self, change):
        try:
            self.i = change["new"]
            self.frame(self.data, self._t())
            self._show_t()
            self.fig.canvas.draw_idle()
        except Exception as err:
            self._report(err)

    def export(self, name, folder="gifs", dpi=120):
        """Teacher only: frames for the LaTeX animate package (current slider values)."""
        export_frames(make_anim(self.fig, lambda i: self.frame(self.data, i / self.nframes * self.Tloop),
                                self.nframes, self.fps), name, folder, dpi)


def period(t, q):
    """Period measured from the upward zero crossings of q(t)."""
    idx = np.where((q[:-1] < 0) & (q[1:] >= 0))[0]
    return np.mean(np.diff(t[idx])) if len(idx) > 1 else np.nan


class Dashpot:
    """Viscous damper drawn between p0 (cylinder side) and p1 (piston side).
    The geometry is fixed by the first call to set() (or after reset()): the piston then moves inside the cylinder."""

    def __init__(self, ax, width=0.01, **kw):
        kw.setdefault("color", "k"); kw.setdefault("lw", 1.2)
        self.w, self.geom = width, None
        self.lines = [ax.plot([], [], **kw)[0] for _ in range(3)]

    def reset(self):
        self.geom = None

    def set(self, p0, p1):
        p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
        L = np.linalg.norm(p1 - p0); u = (p1 - p0) / L; v = np.array([-u[1], u[0]])
        if self.geom is None:
            self.geom = (0.1 * L, 0.65 * L, 0.6 * L)          # stem, cylinder length, piston rod length
        a, cyl, rod = self.geom
        P = lambda s, d=0.0: p0 + s * u + d * v
        stem = np.array([p0, P(a)])
        cylinder = np.array([P(a + cyl, self.w), P(a, self.w), P(a, -self.w), P(a + cyl, -self.w)])
        sp = L - rod
        piston = np.array([P(sp, 0.8 * self.w), P(sp, -0.8 * self.w), P(sp), p1])
        for line, pts in zip(self.lines, [stem, cylinder, piston]):
            line.set_data(pts[:, 0], pts[:, 1])


def double_pendulum_rhs(g, L):
    """Exact equations of a double pendulum with two equal masses and two equal lengths (absolute angles)."""
    def f(_, s):
        t1, t2, w1, w2 = s
        d = t1 - t2
        den = L * (3 - np.cos(2 * d))
        a1 = (-3 * g * np.sin(t1) - g * np.sin(t1 - 2 * t2) - 2 * np.sin(d) * (w2 ** 2 * L + w1 ** 2 * L * np.cos(d))) / den
        a2 = (2 * np.sin(d) * (2 * w1 ** 2 * L + 2 * g * np.cos(t1) + w2 ** 2 * L * np.cos(d))) / den
        return np.array([w1, w2, a1, a2])
    return f


def draw_two_masses(ax, x1, x2, objs=None, gap=0.3, w=0.12, h=0.1, scale=1.0):
    """Two masses between two walls (walls at 0 and 3*gap+2*w); x1, x2 in the units of the drawing."""
    xa = gap + w / 2 + scale * x1
    xb = 2 * gap + 1.5 * w + scale * x2
    xr = 3 * gap + 2 * w
    if objs is None:
        hatch_v = lambda x, s: [ax.plot([x, x], [-0.02, h + 0.06], "k", lw=2)] + \
            [ax.plot([x, x + s], [y, y + 0.02], "k", lw=0.7) for y in np.linspace(-0.02, h + 0.04, 7)]
        hatch_v(0, -0.025); hatch_v(xr, 0.025)
        ax.plot([0, xr], [-0.02, -0.02], "k", lw=1)
        sps = [ax.plot([], [], "k", lw=1.1)[0] for _ in range(3)]
        boxes = [plt.Rectangle((0, 0), w, h, fc="#f4c7c3", ec="k") for _ in range(2)]
        for b in boxes:
            ax.add_patch(b)
        objs = (sps, boxes)
    sps, boxes = objs
    yc = h / 2
    ends = [(0, xa - w / 2), (xa + w / 2, xb - w / 2), (xb + w / 2, xr)]
    for s, (p, q) in zip(sps, ends):
        s.set_data(*spring_xy((p, yc), (q, yc), n=6, width=0.025))
    boxes[0].set_x(xa - w / 2); boxes[1].set_x(xb - w / 2)
    return objs


## Figure 1 – Rayleigh damping
Damping matrix proportional to the mass and stiffness matrices: $[\boldsymbol{C}] = \alpha[\boldsymbol{M}] + \beta[\boldsymbol{K}]$, so that
$\tilde c_i = \alpha\,\tilde m_i + \beta\,\tilde k_i$ and
$$\tilde\zeta_i = \frac{\alpha}{2\omega_i} + \frac{\beta\,\omega_i}{2}.$$
The dashed curves on the modal coordinates are the envelopes $\pm e^{-\tilde\zeta_i\omega_i t}$.
Example: the two masses of Tutorial 4 ($m = 0.5$ kg, $k = 200$ N/m, $\omega_1 = 20$ rad/s, $\omega_2 \approx 34.6$ rad/s), released from $\{q_0\} = a\{1, 0\}^T$ without velocity.

**Observe:** which part of the damping ($\alpha$ or $\beta$) acts on the low and on the high frequencies? With $\beta = 0$, which mode lasts longer? And with $\alpha = 0$?

In [ ]:
#@title Figure 1
def figure_1():
    m, k = 0.5, 200.0
    w = np.sqrt(np.array([1.0, 3.0]) * k / m)                # natural angular frequencies of the two masses
    om = np.linspace(2, 80, 400)
    t = np.linspace(0, 3, 1500)

    def response(z):
        # modal coordinates q_i/q_i(0) (q_i(0) = a/2, zero velocity), then x1/a, x2/a = superposition of the two modes
        q = []
        for i in range(2):
            zi = min(z[i], 0.999); wd = w[i] * np.sqrt(1 - zi ** 2)
            q.append(np.exp(-zi * w[i] * t) * (np.cos(wd * t) + zi / np.sqrt(1 - zi ** 2) * np.sin(wd * t)))
        return 0.5 * (q[0] + q[1]), 0.5 * (q[0] - q[1]), q[0], q[1]

    fig = plt.figure(figsize=(11, 6))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.1, 1, 1], top=0.88, bottom=0.27, hspace=0.45, wspace=0.32)
    axz = fig.add_subplot(gs[:, 0]); ax1 = fig.add_subplot(gs[0, 1]); ax2 = fig.add_subplot(gs[1, 1])
    am1 = fig.add_subplot(gs[0, 2]); am2 = fig.add_subplot(gs[1, 2])
    ax1.set_title("Physical coordinates", fontsize=10); am1.set_title("Modal coordinates", fontsize=10)
    lm, = axz.plot([], [], "--", color=BLUE, lw=1.5, marker="s", markevery=40, ms=5, label=r"mass part $\alpha/2\omega$")
    lk, = axz.plot([], [], ":", color=RED, lw=1.8, marker="^", markevery=40, ms=5, label=r"stiffness part $\beta\omega/2$")
    lt, = axz.plot([], [], "-", color="k", lw=2.2, marker="o", markevery=40, ms=4, label="total")
    pts, = axz.plot([], [], "o", color=GREEN, ms=10, zorder=5, label=r"modes: $\tilde\zeta_1$, $\tilde\zeta_2$")
    for wi in w:
        axz.axvline(wi, color=GREY, lw=0.8, ls="--")
    axz.text(w[0], 1.02, r"$\omega_1$", transform=axz.get_xaxis_transform(), ha="center")
    axz.text(w[1], 1.02, r"$\omega_2$", transform=axz.get_xaxis_transform(), ha="center")
    axz.set_xlim(0, om[-1]); axz.set_ylim(0, 0.3); axz.set_xlabel(r"$\omega$ (rad/s)"); axz.set_ylabel(r"modal damping ratio $\tilde\zeta$")
    axz.grid(alpha=0.3); axz.legend(fontsize=8, loc="upper right")
    c1, = ax1.plot(t, 0 * t, color=BLUE, lw=1.5); c2, = ax2.plot(t, 0 * t, color=RED, lw=1.5, ls=":")
    m1, = am1.plot(t, 0 * t, color=BLUE, lw=1.5); m2, = am2.plot(t, 0 * t, color=RED, lw=1.5, ls=":")
    e1 = [am1.plot(t, 0 * t, "--", color=GREY, lw=0.8)[0] for _ in range(2)]
    e2 = [am2.plot(t, 0 * t, "--", color=GREY, lw=0.8)[0] for _ in range(2)]
    for ax, lab in [(ax1, r"$x_1/a$"), (ax2, r"$x_2/a$"), (am1, r"$\tilde q_1/\tilde q_1(0)$"), (am2, r"$\tilde q_2/\tilde q_2(0)$")]:
        ax.set_xlim(0, t[-1]); ax.set_ylim(-1.05, 1.05); ax.set_ylabel(lab); ax.grid(alpha=0.3); ax.axhline(0, color="k", lw=0.5)
    ax2.set_xlabel(r"$t$ (s)"); am2.set_xlabel(r"$t$ (s)")
    title = fig.suptitle("", fontsize=11)

    fig.subplots_adjust(bottom=0.27)
    sa = Slider(fig.add_axes([0.3, 0.12, 0.4, 0.03]), r"$\alpha$ (s$^{-1}$)", 0.0, 10.0, valinit=1.0, valstep=0.1)
    sb = Slider(fig.add_axes([0.3, 0.06, 0.4, 0.03]), r"$\beta$ (ms)", 0.0, 6.0, valinit=2.0, valstep=0.1)

    def update(_):
        a, b = sa.val, sb.val * 1e-3
        lm.set_data(om, a / (2 * om)); lk.set_data(om, b * om / 2); lt.set_data(om, a / (2 * om) + b * om / 2)
        z = a / (2 * w) + b * w / 2
        pts.set_data(w, z)
        x1, x2, q1, q2 = response(z); c1.set_ydata(x1); c2.set_ydata(x2); m1.set_ydata(q1); m2.set_ydata(q2)
        for j, (ee, zi, wi) in enumerate([(e1, z[0], w[0]), (e2, z[1], w[1])]):
            ee[0].set_ydata(np.exp(-zi * wi * t)); ee[1].set_ydata(-np.exp(-zi * wi * t))
        title.set_text(rf"$\tilde\zeta_1$ = {z[0]:.3f}, $\tilde\zeta_2$ = {z[1]:.3f}   –   decay rates $\tilde\zeta_i\omega_i$: {z[0]*w[0]:.2f} and {z[1]*w[1]:.2f} s$^{{-1}}$")
        fig.canvas.draw_idle()

    sa.on_changed(update); sb.on_changed(update); update(None)
    if INTERACTIVE:
        fig.canvas.header_visible = False
    else:
        print("Static version (alpha = 1, beta = 2 ms): install ipympl to use the sliders.")
    plt.show()
    return sa, sb

fig1 = figure_1()